# 2.2 — Standard vs reasoning model
**Session 2.2 · Demo**

What you'll see: the same logic puzzle answered instantly, then with thinking first. We compare
the answers, the time taken and the tokens used, then read a summary of the model's thinking.

In [1]:
# @title Setup: connect to Claude { display-mode: "form" }
import importlib.util, shutil, subprocess, sys

if importlib.util.find_spec("anthropic") is None:
    installer = (
        ["uv", "pip", "install", "--python", sys.executable]
        if shutil.which("uv")
        else [sys.executable, "-m", "pip", "install"]
    )
    subprocess.check_call([*installer, "-q", "anthropic"])

import html
import os
import re
import time
from getpass import getpass
from itertools import permutations

import anthropic
from IPython.display import HTML, display


def read_api_key():
    """Environment variable, then the Colab secret, then a prompt. The key is never printed."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata

        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        return getpass("Paste your Anthropic API key: ")


client = anthropic.Anthropic(api_key=read_api_key())
SMALL_MODEL = "claude-haiku-5-5"
PRICES = (0.10, 0.50)  # Haiku: dollars per million tokens, (input, output).

INK, SOFT_INK, BLUE, ORANGE, GREEN, RED = "#0b0b0b", "#52514e", "#2a78d6", "#eb6834", "#1baf7a", "#d64545"


def box(label, text, colour=BLUE, background="#fff", note=""):
    """A text box with a coloured left border: blue for answers, orange for thinking."""
    note_html = f'<div style="font-size:12px;color:{SOFT_INK};margin-top:6px">{html.escape(note)}</div>' if note else ""
    return (
        f'<div style="margin:8px 0;padding:10px 14px;border-left:4px solid {colour};background:{background}">'
        f'<div style="font-size:12px;font-weight:600;color:{colour};margin-bottom:4px">{html.escape(label)}</div>'
        f'<div style="color:{INK};white-space:pre-wrap;line-height:1.45">{html.escape(text)}</div>'
        f"{note_html}</div>"
    )


def show_answer(label, text, note=""):
    display(HTML(box(label, text, BLUE, "#fff", note)))


def show_thinking(label, text, note=""):
    display(HTML(box(label, text or "(no thinking returned)", ORANGE, "#fdf3ee", note)))


def verdict(correct):
    """A green tick or a red cross, with the word."""
    colour, word = (GREEN, "✓ correct") if correct else (RED, "✗ wrong")
    return f'<span style="color:{colour};font-weight:700">{word}</span>'


def show_side_by_side(cards):
    """Cards in a row: one per run, with its answer, verdict and numbers."""
    cells = ""
    for card in cards:
        numbers = "".join(
            f'<tr><td style="color:{SOFT_INK};padding:2px 12px 2px 0">{name}</td>'
            f'<td style="color:{INK};text-align:right;font-weight:600">{value}</td></tr>'
            for name, value in card["numbers"].items()
        )
        cells += (
            f'<td style="vertical-align:top;padding:12px 16px;border-left:4px solid {BLUE};background:#fff;width:50%">'
            f'<div style="font-size:15px;font-weight:700;color:{INK};margin-bottom:6px">{html.escape(card["title"])}</div>'
            f'<div style="color:{INK};white-space:pre-wrap;margin-bottom:8px">{html.escape(card["answer"])}</div>'
            f'<div style="margin-bottom:8px">{verdict(card["correct"])}</div>'
            f'<table style="border-collapse:collapse;font-size:13px">{numbers}</table></td>'
        )
    display(HTML(f'<table style="border-collapse:separate;border-spacing:12px 0;width:100%"><tr>{cells}</tr></table>'))


print(f"Ready. Model: {SMALL_MODEL}")

Ready. Model: claude-haiku-5-5


## The puzzle
Each person covers one on-call day. The prompt asks for the answer only, so any working-out
has to happen in the model's thinking.

In [2]:
# @title The puzzle { display-mode: "form" }
PUZZLE = """Seven engineers, Ana, Ben, Chi, Dev, Eli, Fay and Gus, each cover one on-call day
from Monday to Sunday.

1. Exactly two people are on call between Ana and Chi, and Ana comes first.
2. Ben is on call the day right after Dev.
3. Eli is not on Monday or Sunday, and is not on a day next to Ana's.
4. Fay is on call some time after Ben.
5. Gus is on call some time before Ana, and not on a weekend.
6. Fay is not on call the day right before Eli.

Reply with one line only: the seven full names in order from Monday to Sunday, separated by commas."""

show_answer("The puzzle we send", PUZZLE)

Before asking the model, we check every possible order against the six rules, so we know the
right answer.

In [9]:
# @title Check every possible order { display-mode: "form" }
NAMES = ["Ana", "Ben", "Chi", "Dev", "Eli", "Fay", "Gus"]


def fits(order):
    """Python's version of the six rules."""
    day = {name: position for position, name in enumerate(order)}
    return (
        day["Chi"] - day["Ana"] == 3
        and day["Ben"] == day["Dev"] + 1
        and day["Eli"] not in (0, 6)
        and abs(day["Eli"] - day["Ana"]) != 1
        and day["Fay"] > day["Ben"]
        and day["Gus"] < day["Ana"]
        and day["Gus"] < 5
        and day["Fay"] != day["Eli"] - 1
    )


solutions = [order for order in permutations(NAMES) if fits(order)]
display(HTML(
    f'<div style="font-size:16px;color:{INK};margin:6px 0">Tried all <b>5,040</b> orders and found '
    f'<b style="color:{BLUE}">{len(solutions)}</b> solution{"" if len(solutions) == 1 else "s"}.</div>'
    f'<b style="color:{BLUE}">{solutions[0]}.</div>'
))

## Run it both ways
Same model (Haiku), same puzzle. Two runs:
- **instant:** thinking off, and no room to work it out (the answer line only, 60 tokens at most);
- **thinking:** the model thinks first, then answers.

In [4]:
# @title Run it both ways { display-mode: "form" }
INSTANT = {
    "thinking": {"type": "disabled"},
    "system": "Reply with the answer line only. No working.",
    "max_tokens": 60,  # Too short to work the puzzle out in the answer.
}
THINKING = {
    # "summarized" returns a readable summary of the thinking; without it the thinking text is empty.
    "thinking": {"type": "adaptive", "display": "summarized"},
    "max_tokens": 16000,
}


def attempt(label, settings):
    """Run the puzzle once. Returns the thinking text and one card for the display."""
    start = time.perf_counter()
    response = client.messages.create(
        model=SMALL_MODEL, messages=[{"role": "user", "content": PUZZLE}], **settings
    )
    seconds = time.perf_counter() - start

    # The reply is a list of blocks: "thinking" blocks (if any), then the "text" answer.
    answer = "".join(
        block.text for block in response.content if block.type == "text"
    ).strip()
    thinking = "".join(
        block.thinking for block in response.content if block.type == "thinking"
    )
    order = tuple(re.findall(r"\b(?:" + "|".join(NAMES) + r")\b", answer)[-7:])
    usage = response.usage
    cost = (
        usage.input_tokens * PRICES[0] + usage.output_tokens * PRICES[1]
    ) / 1_000_000
    return thinking, {
        "title": label,
        "answer": answer or "(no answer)",
        "correct": order in solutions,
        "numbers": {
            "time": f"{seconds:.1f} s",
            "input tokens": f"{usage.input_tokens:,}",
            "output tokens": f"{usage.output_tokens:,}",
            "cost": f"${cost:.5f}",
        },
    }


_, instant_card = attempt("Instant", INSTANT)
thinking_text, thinking_card = attempt("With thinking", THINKING)
print(SMALL_MODEL)
show_side_by_side([instant_card, thinking_card])

claude-haiku-5-5


time,1.1 s
input tokens,242
output tokens,60
cost,$0.00005
time,7.0 s
input tokens,229
output tokens,"1,553"
cost,$0.00080


**Watch for:** the instant answer is wrong (or cut off mid-sentence), the thinking answer is right.
Thinking costs more time and tokens, because thinking tokens are billed as output. It's worth it on
hard problems, not on easy ones.

## The thinking trace
Claude doesn't show its raw thinking; it returns a readable **summary** of it.
**Watch for:** the model trying each case, ruling some out, and checking the rules again.

In [5]:
# @title Claude's thinking (summary) { display-mode: "form" }
show_thinking("Claude's thinking, as a summary", thinking_text)

## A raw thinking trace, from an open model
Claude never shows its real thinking, only a summary. An open-weight reasoning model shows
everything: **Qwen3-0.6B** writes its reasoning between the markers <think> and </think>, then
answers. It shows each word as it's generated.

**Watch for:** every hesitation is there: "hmm", "wait", "let me check". Each one is a token.

In [6]:
# @title Load the open model (Qwen3-0.6B) { display-mode: "form" }
# @markdown Untick to skip this part (the first run downloads about 1.2 GB).
RUN_OPEN_MODEL = True  # @param {type:"boolean"}

if RUN_OPEN_MODEL:
    missing = [
        package
        for package in ("transformers", "torch")
        if importlib.util.find_spec(package) is None
    ]
    if missing:
        installer = (
            ["uv", "pip", "install", "--python", sys.executable]
            if shutil.which("uv")
            else [sys.executable, "-m", "pip", "install"]
        )
        subprocess.check_call([*installer, "-q", *missing])

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, set_seed

    OPEN_MODEL = "Qwen/Qwen3-0.6B"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    open_tokenizer = AutoTokenizer.from_pretrained(OPEN_MODEL)
    open_model = AutoModelForCausalLM.from_pretrained(
        OPEN_MODEL, dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)
    print(f"Loaded {OPEN_MODEL} on {device}.")
else:
    print("Skipped: the open model isn't loaded.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-0.6B on cpu.


In [7]:
# @title Qwen thinks out loud { display-mode: "form" }
QUESTION = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"  # @param {type:"string"}

if RUN_OPEN_MODEL:
    show_answer("The question", QUESTION)
    display(HTML(f'<div style="font-size:12px;font-weight:600;color:{ORANGE};margin:8px 0 2px">'
                 "Qwen3-0.6B, raw output, word by word (stops after 300 tokens)</div>"))

    # enable_thinking=True asks the model to reason inside <think> ... </think> before it answers.
    prompt = open_tokenizer.apply_chat_template(
        [{"role": "user", "content": QUESTION}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = open_tokenizer(prompt, return_tensors="pt").to(device)

    set_seed(0)
    open_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,  # Qwen's recommended settings for thinking.
        streamer=TextStreamer(open_tokenizer, skip_prompt=True),
    )
else:
    print("Skipped: tick the box in the step above and run both steps.")

<think>
Okay, let's see. There's a problem here where a bat and a ball cost $1.10 in total. Also, the bat costs $1.00 more than the ball. I need to find out how much the ball costs. Hmm, let me think.

First, let me note down the information given:

- Total cost of bat + ball = $1.10
- Cost of bat = cost of ball + $1.00

So, let me assign variables to make this easier. Let me call the cost of the ball 'x'. Then, since the bat is $1.00 more expensive, the cost of the bat would be x + 1.00.

So, according to the total cost, the sum of the ball's cost and the bat's cost equals $1.10. That would be:

x + (x + 1.00) = 1.10

Let me write that equation again:

x + (x + 1.00) = 1.10

Simplify the equation. Combine like terms. The x terms would be 2x, so:

2x + 1.00 = 1.10

Now, subtract 1.00 from both sides to solve for x:

2x = 1.10 - 1.00

Which is:

2x = 0.1


### The same question to Claude
Claude's thinking comes back as a tidy summary, not the raw trace above.

In [8]:
# @title The same question to Claude { display-mode: "form" }
response = client.messages.create(
    model=SMALL_MODEL,
    max_tokens=4000,
    thinking={"type": "adaptive", "display": "summarized"},
    messages=[{"role": "user", "content": QUESTION}],
)
thinking = "".join(block.thinking for block in response.content if block.type == "thinking")
answer = "".join(block.text for block in response.content if block.type == "text").strip()

show_answer("The question", QUESTION)
show_thinking("Claude's thinking, as a summary", thinking)
show_answer("Claude's answer", answer, note=f"{response.usage.output_tokens:,} output tokens, thinking included.")